# 07 · likeを使う事後追加分析（P13 / P14）

Cの主結果とRQ3のlike層別を見た後に計画した**事後追加分析**。主結果・主指標・既存モデル・既存スコア・事実シートを変更しない。

実行時の記録は **DECISIONS.md:235–246（P13）、247–258（P14）**。この記録時点で新しいlike分析のAUC・ラベル別正例率を計算していない。区別するのは「事後計画」と「集計前の定義固定」で、研究全体の事前計画とは呼ばない。

カーネルは **Paper 1 (.venv)**。上からRun Allで、監査・手作りテスト・支持人数確認・P13・P14を順に実行する。実験の再学習はせず、結果は **results/like_analysis/** のみに保存する。共通処理は新規 `src/relisten/like_analysis.py`。

T2/test/既like除外後/全ペアの主D90正例ユーザーが500人未満なら、P13の率・P14のAUCは計算せず継続確認の状態を返す。後続セルも結果集計を表示しない。

mainではP13が226行、P14が238行。登録時点の全文はresults/like_analysis/registered_decisions_snapshot.mdに保存し、元のハッシュと現行P13/P14の一致を検証する。既存の出力は元の実行時のまま保持している。

In [1]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
import json
import os
import subprocess
import polars as pl
from IPython.display import display, Markdown
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/relisten/config.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
from relisten import like_analysis as la
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(14)
pl.Config.set_float_precision(6)
print("実行前記録:", json.dumps(la.REGISTRATION, ensure_ascii=False, indent=2))
print("固定定義:", json.dumps(la.frozen_definition(), ensure_ascii=False, indent=2))

実行前記録: {
  "recorded_at_utc": "2026-10-20T06:05:59.767933+00:00",
  "p13_line": 235,
  "p14_line": 247,
  "end_line": 258,
  "decisions_registered_sha256": "42ea83fd9e90e9282615112097109edda25803a4363cd481a8b268097f5b466c",
  "original_prefix_sha256": "e92aaf2c3f8b9f21d6f8114c29fb3de9a06f0a1b30119595229050517a01a111",
  "original_prefix_bytes": 50248,
  "additional_like_analysis_auc_or_label_rates_computed_before_registration": false
}
固定定義: {
  "classification": "posthoc; recorded before P13/P14 label rates and AUC",
  "P13": "same main L1 cohort; all T1/T2 train/validation/test; prior_like has group precedence",
  "windows_seconds_closed": {
    "like_90d": [
      0,
      7776000
    ],
    "like_24h": [
      0,
      86400
    ]
  },
  "prior_like": "any observed same uid/item like with timestamp strictly before anchor; not current state",
  "likes_routes": "all; repeated and duplicate events use existence OR",
  "P13_reverse_denominators": [
    "all C positives/window event inc

## 0. likes / unlikesの監査（ラベルとの結合前）

列・型・行数・timestampの単位・重複を見る。unlikesは取得しない。存在ORで判定するので、重複likeや同時刻の経路違いで正例数を二重加算しない。既likeは起点前のイベントを観測したことを表し、現在のLike状態を復元したものではない。

In [2]:
audit = la.audit_likes()
display(pl.DataFrame({"item": list(audit), "value": [str(v) for v in audit.values()]}))
print("取り消しは考慮できない。timestampは秒で、5倍しない。")

item,value
str,str
"""path""","""data/raw/flat/50m/likes.parque…"
"""schema""","""{'uid': 'UInt32', 'timestamp':…"
"""rows""","""881456"""
"""users""","""8283"""
"""items""","""181304"""
"""min_timestamp""","""80"""
"""max_timestamp""","""25999970"""
"""all_multiples_of_5""","""True"""
"""unique_full_rows""","""875051"""


取り消しは考慮できない。timestampは秒で、5倍しない。


## 1. 手作りの境界テスト

strict prior `<起点`、起点同時刻、24時間/90日ちょうどと±5秒、同時刻の経路違い・重複、既like優先、別uid/曲、空入力、未来イベントの追加を確認する。seed平均の順序・共通ユーザーへの対応・空AUC・順位比較の欠測にも小さなテストを置く。テスト中のAUCは人工データだけで、実データの集計はまだ行わない。

In [3]:
env = os.environ.copy()
env["PYTHONDONTWRITEBYTECODE"] = "1"
completed = subprocess.run([str(ROOT / ".venv/bin/python"), "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_like_analysis.py"], cwd=ROOT, env=env, text=True, capture_output=True)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
    raise RuntimeError("ラベル/比較の境界テストが失敗したため集計しない")

.................                                                        [100%]
17 passed in 0.83s



## 2. 支持人数の確認 → P13/P14を順番に実行

P13は既存主ラベルの全ペア（L=1・同じ窓で再生日30日以上）。like_24hは[起点,起点+24時間]、like_90dは[起点,起点+90日]。両端を含む。既likeは厳密に起点より前で、3群では最優先。

P14は既likeを除外し、全モデル・D/C/Bで同じペアを使う。全ペアとmatching_snapshotの2通り。matching_snapshotは既存の全経路・全再生率の学習期間出現曲フラグで、モデルごとの語彙ではない。

Dの主定義はlike_90d、like_24hは感度版。8モデルの保存済み生スコアをそのまま使い、向きを変えず、ヘッドを通さない。iALS/BPRは各ユーザー内で3seedのAUCを平均後、ユーザーbootstrap1000回・seed20261016。D−Cは両ラベルでAUCが定義できる共通ユーザーのみ。

In [4]:
result = la.run_like_analysis()
print("実行状態:", result["status"])
display(pl.DataFrame([result["support_gate"]]))
print("実行・既存ファイル保護:", result["execution"])
if result["status"] != "ready":
    display(Markdown("**停止：D90正例ユーザーが500人未満です。TMの継続判断まで、率・AUCの集計を進めていません。**"))

実行前記録・データ監査を確認。Likeのラベルを作ります。


支持人数gate: {'task': 'T2', 'split': 'test', 'selection': 'all_pairs', 'window': 'like_90d', 'd_positive_users': 2952, 'required_users': 500, 'status': 'ready'}


P13 completed


P14 completed: light_features.parquet


P14 completed: ItemKNN_features.parquet


...

P14 completed: BPR_features.parquet


P14 completed: SASRec_features.parquet


実行状態: ready


task,split,selection,window,d_positive_users,required_users,status
str,str,str,str,i64,i64,str
"""T2""","""test""","""all_pairs""","""like_90d""",2952,500,"""ready"""


実行・既存ファイル保護: {'elapsed_seconds': 33.9498219168745, 'protected_files': 2158, 'changed_protected_files': [], 'status': 'ready', 'source_sha256': '561a051ef684f79958a1c64eec70504c52d784ea9e7420e5e3c51984c493eb15', 'metrics_sha256': 'e4508a8ec4a1f284259842dfecc6f151adfb34c8ba90c71a42b65397b9303bf9'}


## 3. P13：likeとC/Bの関連

likeあり/なしは既likeを除いた排他的群。逆向き比率では全C正例と既like除外C正例の分母を分ける。ユーザー内差は、同じ条件でlikeあり/なし両方のペアを持つユーザーのC率差の等重み平均。Cの正負両方を持つことは条件にしていない。train/validation/testをすべてCSVへ保存する。

In [5]:
if result["status"] == "ready":
    for name in ("p13_strata", "p13_reverse", "p13_within_user"):
        print(name, "（test）")
        display(result["tables"][name].filter(pl.col("split") == "test"))

p13_strata （test）


analysis,task,split,window,like_group,pairs,users,c_positives,b_positives,c_positive_rate,b_positive_rate
str,str,str,str,str,i64,i64,i64,i64,f64,f64
"""P13""","""T1""","""test""","""like_90d""","""prior_like""",45949,3149,29616,40048,0.644541,0.871575
"""P13""","""T1""","""test""","""like_90d""","""like""",21469,2778,17371,20245,0.809120,0.942988
"""P13""","""T1""","""test""","""like_90d""","""no_like""",915298,4451,142676,615063,0.155879,0.671981
"""P13""","""T1""","""test""","""like_24h""","""prior_like""",45949,3149,29616,40048,0.644541,0.871575
"""P13""","""T1""","""test""","""like_24h""","""like""",7845,1707,6263,6750,0.798343,0.860421
"""P13""","""T1""","""test""","""like_24h""","""no_like""",928922,4458,153784,628558,0.165551,0.676653
"""P13""","""T2""","""test""","""like_90d""","""prior_like""",14119,2117,10650,11919,0.754303,0.844182
"""P13""","""T2""","""test""","""like_90d""","""like""",37613,2952,28795,33386,0.765560,0.887619
"""P13""","""T2""","""test""","""like_90d""","""no_like""",916152,4813,53742,372339,0.058661,0.406416


p13_reverse （test）


analysis,task,split,window,c_positive_pairs_all,c_positive_pairs_prior_like,c_positive_pairs_nonprior,c_positive_with_window_like_all,c_positive_with_new_like,window_like_fraction_among_all_c_positive,new_like_fraction_among_all_c_positive,new_like_fraction_among_nonprior_c_positive
str,str,str,str,i64,i64,i64,i64,i64,f64,f64,f64
"""P13""","""T1""","""test""","""like_90d""",189663,29616,160047,18099,17371,0.095427,0.091589,0.108537
"""P13""","""T1""","""test""","""like_24h""",189663,29616,160047,6492,6263,0.034229,0.033022,0.039132
"""P13""","""T2""","""test""","""like_90d""",93187,10650,82537,29056,28795,0.311803,0.309002,0.348874
"""P13""","""T2""","""test""","""like_24h""",93187,10650,82537,14841,14787,0.159260,0.158681,0.179156


p13_within_user （test）


analysis,task,split,window,paired_users,mean_like_c_rate,mean_no_like_c_rate,mean_difference,ci_low,ci_high,interpretation
str,str,str,str,i64,f64,f64,f64,f64,f64,str
"""P13""","""T1""","""test""","""like_90d""",2765,0.835725,0.162653,0.673072,0.661635,0.685179,"""関連のみ。因果効果・好みの裏付けとは解釈しない"""
"""P13""","""T1""","""test""","""like_24h""",1701,0.851471,0.169228,0.682242,0.666118,0.697451,"""関連のみ。因果効果・好みの裏付けとは解釈しない"""
"""P13""","""T2""","""test""","""like_90d""",2949,0.824734,0.069399,0.755335,0.743884,0.767453,"""関連のみ。因果効果・好みの裏付けとは解釈しない"""
"""P13""","""T2""","""test""","""like_24h""",2055,0.833928,0.083779,0.750149,0.736057,0.764205,"""関連のみ。因果効果・好みの裏付けとは解釈しない"""


## 4. P14：主D90の生スコアAUCと対応付きD−C

testを主とする。validationも保存済み。AUCの対象人数と、対応差の共通人数を区別する。500人未満は参考値で、陰性ユーザーやlike数だけではAUCの適格人数を代用できない。C/BのAUCもDと同じペア上で計算して保存する。

In [6]:
if result["status"] == "ready":
    for selection in ("all_pairs", "matching_snapshot"):
        print("D90 / test /", selection)
        condition = (pl.col("window") == "like_90d") & (pl.col("split") == "test") & (pl.col("selection") == selection)
        display(result["tables"]["p14_auc"].filter(condition & (pl.col("route") == "D")).select("task", "model", "pairs", "positive_users", "auc_users", "user_auc", "ci_low", "ci_high", "display_status").sort("task", "user_auc", descending=[False, True]))
        display(result["tables"]["p14_paired_d_minus_c"].filter(condition).select("task", "model", "paired_users", "d_auc_common", "c_auc_common", "mean_auc_difference", "difference_ci_low", "difference_ci_high", "display_status").sort("task", "model"))

D90 / test / all_pairs


task,model,pairs,positive_users,auc_users,user_auc,ci_low,ci_high,display_status
str,str,i64,i64,i64,f64,f64,f64,str
"""T1""","""SASRec""",936767,2778,2765,0.546652,0.539549,0.553923,"""通常表示"""
"""T1""","""AudioCosine""",936767,2778,2765,0.533985,0.527029,0.541906,"""通常表示"""
"""T1""","""ACT-R""",936767,2778,2765,0.520100,0.513488,0.527014,"""通常表示"""
"""T1""","""DecayPop""",936767,2778,2765,0.515240,0.508267,0.522769,"""通常表示"""
"""T1""","""ItemKNN""",936767,2778,2765,0.493935,0.487112,0.501039,"""通常表示"""
"""T1""","""MostPop""",936767,2778,2765,0.493814,0.486784,0.501477,"""通常表示"""
"""T1""","""iALS""",936767,2778,2765,0.481550,0.474426,0.487754,"""通常表示"""
"""T1""","""BPR""",936767,2778,2765,0.480616,0.473858,0.487148,"""通常表示"""
"""T2""","""AudioCosine""",953765,2952,2949,0.559262,0.552433,0.565716,"""通常表示"""


task,model,paired_users,d_auc_common,c_auc_common,mean_auc_difference,difference_ci_low,difference_ci_high,display_status
str,str,i64,f64,f64,f64,f64,f64,str
"""T1""","""ACT-R""",2709,0.520243,0.584299,-0.064056,-0.071537,-0.056241,"""通常表示"""
"""T1""","""AudioCosine""",2709,0.534389,0.545137,-0.010748,-0.017129,-0.003799,"""通常表示"""
"""T1""","""BPR""",2709,0.480985,0.499057,-0.018072,-0.024982,-0.011323,"""通常表示"""
"""T1""","""DecayPop""",2709,0.516174,0.537415,-0.021241,-0.028153,-0.014118,"""通常表示"""
"""T1""","""ItemKNN""",2709,0.494729,0.549934,-0.055205,-0.062339,-0.048328,"""通常表示"""
"""T1""","""MostPop""",2709,0.494282,0.530309,-0.036027,-0.043077,-0.028775,"""通常表示"""
"""T1""","""SASRec""",2709,0.547079,0.546425,0.000654,-0.006328,0.007467,"""通常表示"""
"""T1""","""iALS""",2709,0.481806,0.513000,-0.031194,-0.037784,-0.024247,"""通常表示"""
"""T2""","""ACT-R""",2884,0.509933,0.512938,-0.003004,-0.009053,0.002399,"""通常表示"""


D90 / test / matching_snapshot


task,model,pairs,positive_users,auc_users,user_auc,ci_low,ci_high,display_status
str,str,i64,i64,i64,f64,f64,f64,str
"""T1""","""SASRec""",921845,2747,2735,0.559681,0.552242,0.566603,"""通常表示"""
"""T1""","""AudioCosine""",921845,2747,2735,0.531534,0.524134,0.538273,"""通常表示"""
"""T1""","""DecayPop""",921845,2747,2735,0.526677,0.519453,0.534136,"""通常表示"""
"""T1""","""ACT-R""",921845,2747,2735,0.519016,0.511564,0.526590,"""通常表示"""
"""T1""","""ItemKNN""",921845,2747,2735,0.503321,0.496203,0.510434,"""通常表示"""
"""T1""","""MostPop""",921845,2747,2735,0.502956,0.495556,0.509931,"""通常表示"""
"""T1""","""iALS""",921845,2747,2735,0.490826,0.483891,0.497524,"""通常表示"""
"""T1""","""BPR""",921845,2747,2735,0.489904,0.483070,0.496847,"""通常表示"""
"""T2""","""SASRec""",771849,2718,2714,0.564030,0.556232,0.570991,"""通常表示"""


task,model,paired_users,d_auc_common,c_auc_common,mean_auc_difference,difference_ci_low,difference_ci_high,display_status
str,str,i64,f64,f64,f64,f64,f64,str
"""T1""","""ACT-R""",2676,0.518606,0.584609,-0.066002,-0.073378,-0.058759,"""通常表示"""
"""T1""","""AudioCosine""",2676,0.531925,0.545108,-0.013183,-0.020157,-0.006095,"""通常表示"""
"""T1""","""BPR""",2676,0.490364,0.503488,-0.013124,-0.020018,-0.006300,"""通常表示"""
"""T1""","""DecayPop""",2676,0.527284,0.543852,-0.016567,-0.023404,-0.009102,"""通常表示"""
"""T1""","""ItemKNN""",2676,0.504107,0.556951,-0.052844,-0.060071,-0.045679,"""通常表示"""
"""T1""","""MostPop""",2676,0.502799,0.536087,-0.033288,-0.040157,-0.026273,"""通常表示"""
"""T1""","""SASRec""",2676,0.559793,0.553148,0.006645,-0.000020,0.013816,"""通常表示"""
"""T1""","""iALS""",2676,0.491021,0.518348,-0.027327,-0.034266,-0.020481,"""通常表示"""
"""T2""","""ACT-R""",2638,0.515325,0.515078,0.000247,-0.005681,0.006652,"""通常表示"""


## 5. like_24h感度・順位の一致・元のCとの関係

順位相関は8モデルfamilyのKendall τ-bと両側p値。D対C/Bは同じペアで各ラベルのAUC適格ユーザーを使う。D対AのAは、既存149日版モデル・150〜210日複数正解holdoutのNDCG@10を変更せず参照する。Aをvalidation限定や既知曲限定で再評価したものではない。

8モデルでは検出力が低い。非有意なら「一致は確認できなかった」にとどめ、無関係・不一致の証明とはしない。区間・p値は多重比較未補正。

In [7]:
if result["status"] == "ready":
    display(result["tables"]["p14_auc"].filter((pl.col("window") == "like_24h") & (pl.col("split") == "test") & (pl.col("route") == "D")).select("task", "selection", "model", "auc_users", "user_auc", "ci_low", "ci_high", "display_status"))
    display(result["tables"]["p14_kendall"].filter(pl.col("split") == "test"))
    display(result["tables"]["p14_primary_c_context"].sort("task", "selection", "d_rank"))

task,selection,model,auc_users,user_auc,ci_low,ci_high,display_status
str,str,str,i64,f64,f64,f64,str
"""T1""","""all_pairs""","""ACT-R""",1701,0.495052,0.483785,0.506116,"""通常表示"""
"""T1""","""all_pairs""","""AudioCosine""",1701,0.507792,0.497133,0.518125,"""通常表示"""
"""T1""","""all_pairs""","""DecayPop""",1701,0.498309,0.487242,0.508767,"""通常表示"""
"""T1""","""all_pairs""","""MostPop""",1701,0.482924,0.472711,0.493320,"""通常表示"""
"""T1""","""matching_snapshot""","""ACT-R""",1670,0.491970,0.481114,0.502753,"""通常表示"""
"""T1""","""matching_snapshot""","""AudioCosine""",1670,0.506528,0.496383,0.516761,"""通常表示"""
"""T1""","""matching_snapshot""","""DecayPop""",1670,0.508219,0.496993,0.519549,"""通常表示"""
"""T1""","""matching_snapshot""","""MostPop""",1670,0.490665,0.478913,0.501570,"""通常表示"""
"""T2""","""all_pairs""","""ACT-R""",2055,0.506602,0.496552,0.516146,"""通常表示"""


task,split,selection,window,comparison,models,kendall_tau,p_value,minimum_d_auc_users,reference_only,a_reference,interpretation_limit
str,str,str,str,str,i64,f64,f64,i64,bool,str,str
"""T1""","""test""","""all_pairs""","""like_90d""","""D_vs_C""",8,0.571429,0.061012,2765,false,"""same pairs; label-specific AUC…","""8モデルで検出力が低い。非有意から無関係・不一致とは断定しな…"
"""T1""","""test""","""all_pairs""","""like_90d""","""D_vs_B""",8,0.571429,0.061012,2765,false,"""same pairs; label-specific AUC…","""8モデルで検出力が低い。非有意から無関係・不一致とは断定しな…"
"""T1""","""test""","""all_pairs""","""like_90d""","""D_vs_A""",8,-0.285714,0.398760,2765,false,"""unchanged day149/holdout150-21…","""8モデルで検出力が低い。非有意から無関係・不一致とは断定しな…"
"""T1""","""test""","""all_pairs""","""like_24h""","""D_vs_C""",8,0.428571,0.178869,1701,false,"""same pairs; label-specific AUC…","""8モデルで検出力が低い。非有意から無関係・不一致とは断定しな…"
"""T1""","""test""","""all_pairs""","""like_24h""","""D_vs_B""",8,0.428571,0.178869,1701,false,"""same pairs; label-specific AUC…","""8モデルで検出力が低い。非有意から無関係・不一致とは断定しな…"
"""T1""","""test""","""all_pairs""","""like_24h""","""D_vs_A""",8,-0.285714,0.398760,1701,false,"""unchanged day149/holdout150-21…","""8モデルで検出力が低い。非有意から無関係・不一致とは断定しな…"
"""T1""","""test""","""matching_snapshot""","""like_90d""","""D_vs_C""",8,0.500000,0.108681,2735,false,"""same pairs; label-specific AUC…","""8モデルで検出力が低い。非有意から無関係・不一致とは断定しな…"
"""T1""","""test""","""matching_snapshot""","""like_90d""","""D_vs_B""",8,0.500000,0.108681,2735,false,"""same pairs; label-specific AUC…","""8モデルで検出力が低い。非有意から無関係・不一致とは断定しな…"
"""T1""","""test""","""matching_snapshot""","""like_90d""","""D_vs_A""",8,-0.214286,0.548413,2735,false,"""unchanged day149/holdout150-21…","""8モデルで検出力が低い。非有意から無関係・不一致とは断定しな…"


task,selection,model,D,C,B,primary_c_auc,primary_c_rank,same_pairs_c_rank,d_rank
str,str,str,f64,f64,f64,f64,f64,f64,f64
"""T1""","""all_pairs""","""SASRec""",0.546652,0.541917,0.600205,0.544801,4.000000,4.000000,1.000000
"""T1""","""all_pairs""","""AudioCosine""",0.533985,0.550128,0.553867,0.550597,3.000000,3.000000,2.000000
"""T1""","""all_pairs""","""ACT-R""",0.520100,0.596166,0.597349,0.616118,1.000000,1.000000,3.000000
"""T1""","""all_pairs""","""DecayPop""",0.515240,0.539243,0.545676,0.536446,5.000000,5.000000,4.000000
"""T1""","""all_pairs""","""ItemKNN""",0.493935,0.554505,0.565646,0.559807,2.000000,2.000000,5.000000
"""T1""","""all_pairs""","""MostPop""",0.493814,0.532074,0.536460,0.529671,6.000000,6.000000,6.000000
"""T1""","""all_pairs""","""iALS""",0.481550,0.510647,0.547887,0.508560,7.000000,7.000000,7.000000
"""T1""","""all_pairs""","""BPR""",0.480616,0.498187,0.542061,0.496336,8.000000,8.000000,8.000000
"""T1""","""matching_snapshot""","""SASRec""",0.559681,0.548568,0.604356,0.544801,4.000000,4.000000,1.000000


## 6. 言えること／言えないこと

- 観測ログ内のlikeとC/Bの**関連**、保存スコアによる補助ラベルDの識別性能、同じユーザーでのD−C差は報告できる。
- likeした曲がライブラリに入り、自発再生しやすくなる可能性がある。Yambdaではライブラリ・検索などの自発再生の入口を区別できない。因果や「好みの裏付け」とは断定できない。
- like_90dとCは同じ追跡期間中のイベントで、どちらが先かも固定していない。ユーザー内比較でも曲・時刻等の交絡や逆因果は残る。
- unlikesがないため取消は考慮できない。Likeの最終イベント時刻は収録終了の証明ではなく、Like完全観測を新たに保証したものではない。
- Dと元のCではラベル・既like除外・AUC適格ユーザーが変わる。順位が違っても主Cの結果を書き換えたり、モデルを選び直したりしない。

詳細な結果・制約・出典はresults/like_analysis/REPORT.md、固定定義・入力SHA256・実行前記録はprotocol.json。元のコード・結果・事実シートは変更していない。

In [8]:
report = la.OUTPUT / "REPORT.md"
print("レポート:", report)
print("CSV一覧:", [p.name for p in sorted(la.OUTPUT.glob("*.csv"))])
print("P13実行前記録235行、P14実行前記録247行。既存記録を消さず追記のみ。")

レポート: /Users/macuser/dev/paper1/results/like_analysis/REPORT.md
CSV一覧: ['p13_reverse.csv', 'p13_strata.csv', 'p13_user_differences.csv', 'p13_within_user.csv', 'p14_auc.csv', 'p14_kendall.csv', 'p14_paired_d_minus_c.csv', 'p14_paired_user_auc.csv', 'p14_primary_c_context.csv', 'p14_rank_details.csv', 'support_gate.csv']
P13実行前記録235行、P14実行前記録247行。既存記録を消さず追記のみ。
